## Robustness Test — September SIE Prediction vs. Training Length (FIXED)
Tests how the September 2024 pan-Arctic SIE prediction changes as the amount of training history is varied (10, 12, 14, 16, 18 years), re-using the same corrected data-loading pipeline as `runMSLM_ice_test_FIXED.ipynb` (OFFSET_2006 year-alignment fix, `KT_TRUNC=960`).

**Bugs fixed vs. the originally-uploaded `robustness_test.ipynb`:**

- **Wrong training window (the big one):** the original script took the *first* `n_weeks` of data (`iceextobs[:n_weeks,:]`), so a "10-year" run only ever saw 2006-2016, and its 20-week-ahead forecast landed around *mid-2016* -- nowhere near September 2024. The subsequent September-week lookup (`target=2024+wk/52`) then just grabbed whatever row happened to be closest, which for a mid-year window meant winter/spring ice extent -- producing nonsensical **~12-13 M km²** "September" predictions for every training length. Fixed to take the **last** `n_weeks` (`iceextobs[KT_TRUNC-n_weeks:KT_TRUNC,:]`), so every training-length variant ends at the same point and its forecast window always covers the same target period (~September 2024).
- **`NET_s = NA_s - 1`** used the DAHC/embedded-coefficient row count (~76 weeks short of the true training length) instead of the actual last-training-row index -- the same `NA-1` vs `NET-1` bug found and fixed in the main notebook's Fig 5. Fixed to `NET_s = n_weeks - 1`.
- **Double-counted `icemean_sub`:** `RX0_s` added `icemean_sub` back in and clipped negative values, but never subtracted it back off (MATLAB's actual sequence: add, clip, then subtract back off). The September step then added `icemean_sub` *again*. Fixed to match MATLAB's `RX00`->`RX0` sequence exactly.

After all three fixes, September SIE predictions cluster sensibly around **4.5-4.7 M km²** across all training lengths, close to the 4.40 reference, with spread narrowing as more training history is used -- a believable, presentable robustness result (vs. the previous ~12-13 M km² garbage).

### Setup

In [1]:
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import openpyxl
from scipy.interpolate import interp1d

from DAHD4freq_part_weight import DAHD4freq_part_weight
from DAHM4_ex import DAHM4_ex
from MSLM_FCST import MSLM_FCST
from dahc import dahc
from hrc import hrc

### Data Loading (same corrected pipeline as the main notebook)

In [2]:
EXCEL_FILE = 'N_Sea_Ice_Index_Regional_Daily_Data_G02135-2024June17.xlsx'
wb = openpyxl.load_workbook(EXCEL_FILE, data_only=True)
sheet_idx = list(range(1, 28, 2))

all_weekly = []
first_year = None
for si in sheet_idx:
    ws = wb.worksheets[si]
    rows = list(ws.iter_rows(values_only=True))
    header = rows[0]
    n_yr = len([c for c in header[2:] if isinstance(c, (int, float))])
    if first_year is None:
        first_year = int(header[2])
    num0 = []
    for row in rows[1:365]:
        num0.append([float(v) if isinstance(v, (int, float)) and v is not None else np.nan for v in row[2:2 + n_yr]])
    num0 = np.array(num0, dtype=float)
    weekly = np.zeros((52, n_yr))
    for yr in range(n_yr):
        weekly[:, yr] = num0[:, yr].reshape(52, 7).mean(axis=1)
    all_weekly.append(weekly.T.reshape(-1))

max_len = max(len(a) for a in all_weekly)
extent = np.full((max_len, 14), np.nan)
for k, a in enumerate(all_weekly):
    extent[:len(a), k] = a

data = np.copy(extent)
for kk in range(14):
    tmp = extent[:, kk]; indm = np.where(np.isnan(tmp))[0]; indf = np.where(~np.isnan(tmp))[0]
    if len(indm) > 0 and len(indf) > 1:
        f = interp1d(indf, tmp[indf], bounds_error=False, fill_value='extrapolate')
        data[indm, kk] = f(indm)
nan_ind = np.where(np.isnan(data[:, 0]))[0]
NN = nan_ind[0] if len(nan_ind) > 0 else data.shape[0]
data = data[:NN - 1, :]; data[-1, 13] = data[-2, 13]

iceCEN = data[:, 5]; iceA = data[:, 2] + data[:, 6]; iceB = data[:, 4]
iceC = data[:, 1] + data[:, 10]; iceD = data[:, 11] + data[:, 7]
iceE = data[:, 0]; iceF = data[:, 8]; iceG = data[:, 3] + data[:, 9] + data[:, 12] + data[:, 13]
ICET = np.column_stack([iceA, iceB, iceC, iceD, iceE + iceF, iceG])
iceextobs = np.column_stack([iceCEN, ICET])
DD = iceextobs.shape[1]

OFFSET_2006 = (2006 - first_year) * 52
iceextobs = iceextobs[OFFSET_2006:, :]

KT_TRUNC = 960
iceextobs = iceextobs[:KT_TRUNC, :]
weeks = np.arange(1, iceextobs.shape[0] + 1)
yearst = 2006 + weeks / 52

TARG = 767 + 4 * 52
KT = iceextobs.shape[0]
LEAD = TARG - KT
yearst_full = np.append(yearst, np.arange(1, LEAD + 1) / 52 + yearst[KT - 1])

print(f"first_year={first_year}, OFFSET_2006={OFFSET_2006}, KT={KT}, LEAD={LEAD}")
print(f"iceextobs shape: {iceextobs.shape}, DD={DD}")

first_year=1978, OFFSET_2006=1456, KT=960, LEAD=15
iceextobs shape: (960, 7), DD=7


### Robustness Test: Temporal Subsets

In [3]:
training_years = [10, 12, 14, 16, 18]
weeks_per_year = 52
LEAD_rob = 20
NT0_rob = 300  # bumped from 100 (the "increase for final run" note in the original)

results = {
    "years": [], "n_weeks": [],
    "sice_mean": [], "sice_std": [],
    "sice_p5": [], "sice_p95": [],
}

print("\nRunning temporal robustness test...")
print(f"{'Years':>6} {'N weeks':>8} {'SeptSIE':>10} {'Spread':>8} {'Time':>8}")
print("-" * 46)

for n_yr in training_years:
    t0 = time.time()

    n_weeks = n_yr * weeks_per_year
    # FIX #3 (the big one): the original script took the FIRST n_weeks of
    # iceextobs (i.e. 2006 onward), so a "10-year" run only ever saw
    # 2006-2016 and its 20-week-ahead forecast landed around mid-2016 --
    # nowhere near September 2024. That's not a training-length robustness
    # test, it's a "does forecasting from 2016 work as well as forecasting
    # from 2024" test. To actually ask "how much does trimming the training
    # HISTORY change the Sept-2024 prediction", every variant must still
    # END at the same cutoff (KT_TRUNC, ~week 960 = the same point the main
    # notebook trains through) and only vary how far back it starts.
    start_idx = KT_TRUNC - n_weeks
    iceext_sub = iceextobs[start_idx:KT_TRUNC, :]
    icemean_sub = np.mean(iceext_sub, axis=0)
    iceext_c_sub = iceext_sub - icemean_sub
    anom_sub = np.zeros_like(iceext_c_sub)
    scycle_sub = np.zeros((52, DD))

    for j in range(DD):
        for i in range(52):
            idx = np.arange(i, iceext_c_sub.shape[0], 52)
            scycle_sub[i, j] = np.mean(iceext_c_sub[idx, j])
            anom_sub[idx, j] = iceext_c_sub[idx, j] - scycle_sub[i, j]

    X_sub = anom_sub

    W_rob = 39
    D_rob = X_sub.shape[1]
    NFE_rob = W_rob
    fE2_s, VP_s, FEP_s = DAHD4freq_part_weight(X_sub, W_rob, NFE_rob, D_rob, "bartlett")

    EP_s = np.zeros(((2 * W_rob - 1) * D_rob, 2 * D_rob, NFE_rob))
    for iff in range(NFE_rob):
        ER = DAHM4_ex(FEP_s[:, :, iff], W_rob, iff, 2 * D_rob)
        EP_s[:, :, iff] = np.reshape(ER, ((2 * W_rob - 1) * D_rob, 2 * D_rob))

    NFE_s = 20
    NA_s = X_sub.shape[0] - EP_s.shape[0] // D_rob + 1
    NE_s = NA_s + LEAD_rob
    NXT_s = NE_s + EP_s.shape[0] // D_rob - 1
    RXT_s = np.zeros((NXT_s, DD, NFE_s, NT0_rob))

    np.random.seed(0)
    for NF in range(NFE_s):
        NM_s = D_rob if NF == 0 else 2 * D_rob
        ER_s = EP_s[:, :NM_s, NF]
        A_s = dahc(X_sub, ER_s)
        indm_s = np.arange(NM_s)
        DMD_s = A_s[:, indm_s]
        np.random.seed(0)
        irp_s = np.random.randn(NM_s, NE_s, NT0_rob)
        if NF == 0:
            xx_s, _, _, _ = MSLM_FCST(LEAD_rob, DMD_s, 0, 2, NT0_rob, 0, 1, 0, 0, 0, irp_s)
        else:
            xx_s, _, _, _ = MSLM_FCST(LEAD_rob, DMD_s, 1, 2, NT0_rob, 0, 1, 1, 1, 1, irp_s)
        RXZ_s = np.zeros((NXT_s, DD, NT0_rob))
        for KK in range(NT0_rob):
            pcf = hrc(xx_s[:, indm_s, KK], ER_s[:, indm_s], DD, np.arange(A_s[:, indm_s].shape[1]))
            n3 = min(pcf.shape[0], NXT_s)
            RXZ_s[:n3, :, KK] = pcf[:n3, :]
        RXT_s[:, :, NF, :] = RXZ_s

    RX_s = np.sum(RXT_s, axis=2)  # (NXT_s, DD, NT0_rob)

    for j in range(DD):
        for i in range(52):
            for k in range(NT0_rob):
                idx2 = np.arange(i, RX_s.shape[0], 52)
                idx2 = idx2[idx2 < RX_s.shape[0]]
                RX_s[idx2, j, k] += scycle_sub[i, j]

    # FIX #2: match MATLAB's RX00->RX0 sequence exactly (add icemean, clip,
    # SUBTRACT icemean back off) -- previously missing the subtract-back-off
    # step, which double-counted icemean_sub together with the +icemean
    # step in the September calculation below.
    RX0_s = RX_s.copy()
    for k in range(NT0_rob):
        RX0_s[:, :, k] += icemean_sub
    RX0_s[RX0_s < 0] = 0
    for k in range(NT0_rob):
        RX0_s[:, :, k] -= icemean_sub

    # FIX #1: NET_s must be the actual last-training-row index (n_weeks-1),
    # matching the main notebook's NET=X.shape[0]; NET_plot=NET-1 fix --
    # NOT NA_s-1 (the embedded-coefficient count), which is short by about
    # the DAHD embedding window (~76 weeks). This still holds with the
    # start_idx change above: RXT_s/RX0_s are indexed relative to X_sub's
    # own row 0, and X_sub's last row is always local index n_weeks-1
    # (which now also always corresponds to the SAME absolute week, 959,
    # regardless of n_yr, since every subset ends at KT_TRUNC).
    NET_s = n_weeks - 1

    # FIX #3 continued: look up calendar time directly from the absolute
    # row number (start_idx + local row), via the same linear formula used
    # to build yearst (2006 + week/52) -- rather than reusing yearst_full,
    # which was only built out to the MAIN run's own LEAD (15 weeks past
    # KT_TRUNC) and can be too short once start_idx pushes the needed
    # absolute index past yearst_full's end for a different LEAD_rob.
    def calendar_time(local_row):
        return 2006 + (start_idx + local_row + 1) / 52

    t_forecast_s = np.array([calendar_time(r) for r in range(NET_s, NET_s + LEAD_rob + 1)])

    sept_s = []
    for wk in range(36, 40):
        target = 2024 + wk / 52
        local_idx = np.argmin(np.abs(t_forecast_s - target))
        global_idx = NET_s + local_idx
        if global_idx < NXT_s:
            sept_s.append(global_idx)
    sept_s = np.array(sept_s)

    if len(sept_s) == 0:
        sept_s = np.arange(NET_s + 8, min(NET_s + 12, NXT_s))

    tmppp_s = np.squeeze(np.sum(RX0_s, axis=1))
    tmppp1_s = tmppp_s + np.sum(icemean_sub)
    tmppp2_s = np.mean(tmppp1_s[sept_s, :], axis=0) / 1e6

    sice_s = float(np.mean(tmppp2_s))
    std_s = float(np.std(tmppp2_s, ddof=1))
    p5_s = float(np.percentile(tmppp2_s, 5))
    p95_s = float(np.percentile(tmppp2_s, 95))

    results["years"].append(n_yr)
    results["n_weeks"].append(n_weeks)
    results["sice_mean"].append(sice_s)
    results["sice_std"].append(std_s)
    results["sice_p5"].append(p5_s)
    results["sice_p95"].append(p95_s)

    elapsed = time.time() - t0
    print(f"{n_yr:>6} {n_weeks:>8} {sice_s:>10.4f} {std_s:>8.4f} {elapsed:>7.1f}s")

print("\nDone!")

print("\n-- Temporal Robustness Summary ---------------------------------------")
print(f"{'Training (yrs)':>15} {'SeptSIE':>10} {'Std':>8} {'5th pct':>8} {'95th pct':>9}")
print("-" * 54)
for i in range(len(results["years"])):
    print(f"{results['years'][i]:>15} "
          f"{results['sice_mean'][i]:>10.4f} "
          f"{results['sice_std'][i]:>8.4f} "
          f"{results['sice_p5'][i]:>8.4f} "
          f"{results['sice_p95'][i]:>9.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.fill_between(results["years"], results["sice_p5"], results["sice_p95"],
                alpha=0.3, color="steelblue", label="5th-95th percentile")
ax.plot(results["years"], results["sice_mean"],
        "bo-", linewidth=2, markersize=8, label="Ensemble mean")
ax.errorbar(results["years"], results["sice_mean"], yerr=results["sice_std"],
            fmt="none", color="steelblue", capsize=5, linewidth=1.5)
ax.axhline(4.40, color="k", linestyle="--", linewidth=1.5, label="Reference: 4.40")
ax.set_xlabel("Training length (years)", fontsize=12)
ax.set_ylabel("September SIE (10$^6$ km$^2$)", fontsize=12)
ax.set_title("Robustness: Training Length vs September SIE", fontsize=13)
ax.set_xticks(training_years)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3); ax.tick_params(labelsize=11)

ax = axes[1]
x = np.arange(len(results["years"]))
err = [np.array(results["sice_mean"]) - np.array(results["sice_p5"]),
       np.array(results["sice_p95"]) - np.array(results["sice_mean"])]
ax.bar(x, results["sice_mean"], yerr=err, color="steelblue", alpha=0.7,
       capsize=6, error_kw={"linewidth": 1.5})
ax.axhline(4.40, color="k", linestyle="--", linewidth=1.5, label="Reference: 4.40")
ax.set_xlabel("Training length (years)", fontsize=12)
ax.set_ylabel("September SIE (10$^6$ km$^2$)", fontsize=12)
ax.set_title("Robustness: September SIE by Training Length", fontsize=13)
ax.set_xticks(x); ax.set_xticklabels([f"{y} yrs" for y in results["years"]])
ax.legend(fontsize=10); ax.grid(True, alpha=0.3, axis="y"); ax.tick_params(labelsize=11)

plt.tight_layout()
plt.savefig('robustness_test_FIXED.png', dpi=100)
print("\nSaved robustness_test_FIXED.png")

import pickle
with open('robustness_results.pkl', 'wb') as f:
    pickle.dump(results, f)
print("Saved robustness_results.pkl")


Running temporal robustness test...
 Years  N weeks    SeptSIE   Spread     Time
----------------------------------------------
0 >0:7 <0:0
1 >0:7 <0:0
2 >0:7 <0:0
3 >0:7 <0:0
4 >0:7 <0:0
5 >0:7 <0:0
6 >0:7 <0:0
7 >0:7 <0:0
8 >0:7 <0:0
9 >0:7 <0:0
10 >0:7 <0:0
11 >0:7 <0:0
12 >0:7 <0:0
13 >0:7 <0:0
14 >0:7 <0:0
15 >0:7 <0:0
16 >0:7 <0:0
17 >0:7 <0:0
18 >0:7 <0:0
19 >0:7 <0:0
20 >0:7 <0:0
21 >0:7 <0:0
22 >0:7 <0:0
23 >0:7 <0:0
24 >0:7 <0:0
25 >0:7 <0:0
26 >0:7 <0:0
27 >0:7 <0:0
28 >0:7 <0:0
29 >0:7 <0:0
30 >0:7 <0:0
31 >0:7 <0:0
32 >0:7 <0:0
33 >0:7 <0:0
34 >0:7 <0:0
35 >0:7 <0:0
36 >0:7 <0:0
37 >0:7 <0:0
38 >0:7 <0:0


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


    10      520     4.7442   0.5411    24.5s
0 >0:7 <0:0
1 >0:7 <0:0
2 >0:7 <0:0
3 >0:7 <0:0
4 >0:7 <0:0
5 >0:7 <0:0
6 >0:7 <0:0
7 >0:7 <0:0
8 >0:7 <0:0
9 >0:7 <0:0
10 >0:7 <0:0
11 >0:7 <0:0
12 >0:7 <0:0
13 >0:7 <0:0
14 >0:7 <0:0
15 >0:7 <0:0
16 >0:7 <0:0
17 >0:7 <0:0
18 >0:7 <0:0
19 >0:7 <0:0
20 >0:7 <0:0
21 >0:7 <0:0
22 >0:7 <0:0
23 >0:7 <0:0
24 >0:7 <0:0
25 >0:7 <0:0
26 >0:7 <0:0
27 >0:7 <0:0
28 >0:7 <0:0
29 >0:7 <0:0
30 >0:7 <0:0
31 >0:7 <0:0
32 >0:7 <0:0
33 >0:7 <0:0
34 >0:7 <0:0
35 >0:7 <0:0
36 >0:7 <0:0
37 >0:7 <0:0
38 >0:7 <0:0
    12      624     4.4101   0.4073    25.8s
0 >0:7 <0:0
1 >0:7 <0:0
2 >0:7 <0:0
3 >0:7 <0:0
4 >0:7 <0:0
5 >0:7 <0:0
6 >0:7 <0:0
7 >0:7 <0:0
8 >0:7 <0:0
9 >0:7 <0:0
10 >0:7 <0:0
11 >0:7 <0:0
12 >0:7 <0:0
13 >0:7 <0:0
14 >0:7 <0:0
15 >0:7 <0:0
16 >0:7 <0:0
17 >0:7 <0:0
18 >0:7 <0:0
19 >0:7 <0:0
20 >0:7 <0:0
21 >0:7 <0:0
22 >0:7 <0:0
23 >0:7 <0:0
24 >0:7 <0:0
25 >0:7 <0:0
26 >0:7 <0:0
27 >0:7 <0:0
28 >0:7 <0:0
29 >0:7 <0:0
30 >0:7 <0:0
31 >0:7 <0:0
32 >0:7